In [ ]:
!pip install evaluate
!pip install wandb
!pip install bioc
!pip install Seqeval
!pip install imblearn

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from enum import Enum
import logging
from pathlib import Path
from types import SimpleNamespace
from typing import TYPE_CHECKING, Dict, Iterable, List, Tuple

import datasets

if TYPE_CHECKING:
    import bioc

logger = logging.getLogger(__name__)


BigBioValues = SimpleNamespace(NULL="<BB_NULL_STR>")


@dataclass
class BigBioConfig(datasets.BuilderConfig):
    """BuilderConfig for BigBio."""

    name: str = None
    version: datasets.Version = None
    description: str = None
    schema: str = None
    subset_id: str = None


class Tasks(Enum):
    NAMED_ENTITY_RECOGNITION = "NER"
    NAMED_ENTITY_DISAMBIGUATION = "NED"
    EVENT_EXTRACTION = "EE"
    RELATION_EXTRACTION = "RE"
    COREFERENCE_RESOLUTION = "COREF"
    QUESTION_ANSWERING = "QA"
    TEXTUAL_ENTAILMENT = "TE"
    SEMANTIC_SIMILARITY = "STS"
    TEXT_PAIRS_CLASSIFICATION = "TXT2CLASS"
    PARAPHRASING = "PARA"
    TRANSLATION = "TRANSL"
    SUMMARIZATION = "SUM"
    TEXT_CLASSIFICATION = "TXTCLASS"


entailment_features = datasets.Features(
    {
        "id": datasets.Value("string"),
        "premise": datasets.Value("string"),
        "hypothesis": datasets.Value("string"),
        "label": datasets.Value("string"),
    }
)

pairs_features = datasets.Features(
    {
        "id": datasets.Value("string"),
        "document_id": datasets.Value("string"),
        "text_1": datasets.Value("string"),
        "text_2": datasets.Value("string"),
        "label": datasets.Value("string"),
    }
)

qa_features = datasets.Features(
    {
        "id": datasets.Value("string"),
        "question_id": datasets.Value("string"),
        "document_id": datasets.Value("string"),
        "question": datasets.Value("string"),
        "type": datasets.Value("string"),
        "choices": [datasets.Value("string")],
        "context": datasets.Value("string"),
        "answer": datasets.Sequence(datasets.Value("string")),
    }
)

text_features = datasets.Features(
    {
        "id": datasets.Value("string"),
        "document_id": datasets.Value("string"),
        "text": datasets.Value("string"),
        "labels": [datasets.Value("string")],
    }
)

text2text_features = datasets.Features(
    {
        "id": datasets.Value("string"),
        "document_id": datasets.Value("string"),
        "text_1": datasets.Value("string"),
        "text_2": datasets.Value("string"),
        "text_1_name": datasets.Value("string"),
        "text_2_name": datasets.Value("string"),
    }
)

kb_features = datasets.Features(
    {
        "id": datasets.Value("string"),
        "document_id": datasets.Value("string"),
        "passages": [
            {
                "id": datasets.Value("string"),
                "type": datasets.Value("string"),
                "text": datasets.Sequence(datasets.Value("string")),
                "offsets": datasets.Sequence([datasets.Value("int32")]),
            }
        ],
        "entities": [
            {
                "id": datasets.Value("string"),
                "type": datasets.Value("string"),
                "text": datasets.Sequence(datasets.Value("string")),
                "offsets": datasets.Sequence([datasets.Value("int32")]),
                "normalized": [
                    {
                        "db_name": datasets.Value("string"),
                        "db_id": datasets.Value("string"),
                    }
                ],
            }
        ],
        "events": [
            {
                "id": datasets.Value("string"),
                "type": datasets.Value("string"),
                # refers to the text_bound_annotation of the trigger
                "trigger": {
                    "text": datasets.Sequence(datasets.Value("string")),
                    "offsets": datasets.Sequence([datasets.Value("int32")]),
                },
                "arguments": [
                    {
                        "role": datasets.Value("string"),
                        "ref_id": datasets.Value("string"),
                    }
                ],
            }
        ],
        "coreferences": [
            {
                "id": datasets.Value("string"),
                "entity_ids": datasets.Sequence(datasets.Value("string")),
            }
        ],
        "relations": [
            {
                "id": datasets.Value("string"),
                "type": datasets.Value("string"),
                "arg1_id": datasets.Value("string"),
                "arg2_id": datasets.Value("string"),
                "normalized": [
                    {
                        "db_name": datasets.Value("string"),
                        "db_id": datasets.Value("string"),
                    }
                ],
            }
        ],
    }
)


TASK_TO_SCHEMA = {
    Tasks.NAMED_ENTITY_RECOGNITION.name: "KB",
    Tasks.NAMED_ENTITY_DISAMBIGUATION.name: "KB",
    Tasks.EVENT_EXTRACTION.name: "KB",
    Tasks.RELATION_EXTRACTION.name: "KB",
    Tasks.COREFERENCE_RESOLUTION.name: "KB",
    Tasks.QUESTION_ANSWERING.name: "QA",
    Tasks.TEXTUAL_ENTAILMENT.name: "TE",
    Tasks.SEMANTIC_SIMILARITY.name: "PAIRS",
    Tasks.TEXT_PAIRS_CLASSIFICATION.name: "PAIRS",
    Tasks.PARAPHRASING.name: "T2T",
    Tasks.TRANSLATION.name: "T2T",
    Tasks.SUMMARIZATION.name: "T2T",
    Tasks.TEXT_CLASSIFICATION.name: "TEXT",
}

SCHEMA_TO_TASKS = defaultdict(set)
for task, schema in TASK_TO_SCHEMA.items():
    SCHEMA_TO_TASKS[schema].add(task)
SCHEMA_TO_TASKS = dict(SCHEMA_TO_TASKS)

VALID_TASKS = set(TASK_TO_SCHEMA.keys())
VALID_SCHEMAS = set(TASK_TO_SCHEMA.values())

SCHEMA_TO_FEATURES = {
    "KB": kb_features,
    "QA": qa_features,
    "TE": entailment_features,
    "T2T": text2text_features,
    "TEXT": text_features,
    "PAIRS": pairs_features,
}


def get_texts_and_offsets_from_bioc_ann(ann: "bioc.BioCAnnotation") -> Tuple:

    offsets = [(loc.offset, loc.offset + loc.length) for loc in ann.locations]

    text = ann.text

    if len(offsets) > 1:
        i = 0
        texts = []
        for start, end in offsets:
            chunk_len = end - start
            texts.append(text[i : chunk_len + i])
            i += chunk_len
            while i < len(text) and text[i] == " ":
                i += 1
    else:
        texts = [text]

    return offsets, texts


def remove_prefix(a: str, prefix: str) -> str:
    if a.startswith(prefix):
        a = a[len(prefix) :]
    return a


def parse_brat_file(
    txt_file: Path,
    annotation_file_suffixes: List[str] = None,
    parse_notes: bool = False,
) -> Dict:
    """
    Parse a brat file into the schema defined below.
    `txt_file` should be the path to the brat '.txt' file you want to parse, e.g. 'data/1234.txt'
    Assumes that the annotations are contained in one or more of the corresponding '.a1', '.a2' or '.ann' files,
    e.g. 'data/1234.ann' or 'data/1234.a1' and 'data/1234.a2'.
    Will include annotator notes, when `parse_notes == True`.
    brat_features = datasets.Features(
        {
            "id": datasets.Value("string"),
            "document_id": datasets.Value("string"),
            "text": datasets.Value("string"),
            "text_bound_annotations": [  # T line in brat, e.g. type or event trigger
                {
                    "offsets": datasets.Sequence([datasets.Value("int32")]),
                    "text": datasets.Sequence(datasets.Value("string")),
                    "type": datasets.Value("string"),
                    "id": datasets.Value("string"),
                }
            ],
            "events": [  # E line in brat
                {
                    "trigger": datasets.Value(
                        "string"
                    ),  # refers to the text_bound_annotation of the trigger,
                    "id": datasets.Value("string"),
                    "type": datasets.Value("string"),
                    "arguments": datasets.Sequence(
                        {
                            "role": datasets.Value("string"),
                            "ref_id": datasets.Value("string"),
                        }
                    ),
                }
            ],
            "relations": [  # R line in brat
                {
                    "id": datasets.Value("string"),
                    "head": {
                        "ref_id": datasets.Value("string"),
                        "role": datasets.Value("string"),
                    },
                    "tail": {
                        "ref_id": datasets.Value("string"),
                        "role": datasets.Value("string"),
                    },
                    "type": datasets.Value("string"),
                }
            ],
            "equivalences": [  # Equiv line in brat
                {
                    "id": datasets.Value("string"),
                    "ref_ids": datasets.Sequence(datasets.Value("string")),
                }
            ],
            "attributes": [  # M or A lines in brat
                {
                    "id": datasets.Value("string"),
                    "type": datasets.Value("string"),
                    "ref_id": datasets.Value("string"),
                    "value": datasets.Value("string"),
                }
            ],
            "normalizations": [  # N lines in brat
                {
                    "id": datasets.Value("string"),
                    "type": datasets.Value("string"),
                    "ref_id": datasets.Value("string"),
                    "resource_name": datasets.Value(
                        "string"
                    ),  # Name of the resource, e.g. "Wikipedia"
                    "cuid": datasets.Value(
                        "string"
                    ),  # ID in the resource, e.g. 534366
                    "text": datasets.Value(
                        "string"
                    ),  # Human readable description/name of the entity, e.g. "Barack Obama"
                }
            ],
            ### OPTIONAL: Only included when `parse_notes == True`
            "notes": [  # # lines in brat
                {
                    "id": datasets.Value("string"),
                    "type": datasets.Value("string"),
                    "ref_id": datasets.Value("string"),
                    "text": datasets.Value("string"),
                }
            ],
        },
        )
    """

    example = {}
    example["document_id"] = txt_file.with_suffix("").name
    with txt_file.open() as f:
        example["text"] = f.read()

    # If no specific suffixes of the to-be-read annotation files are given - take standard suffixes
    # for event extraction
    if annotation_file_suffixes is None:
        annotation_file_suffixes = [".a1", ".a2", ".ann"]

    if len(annotation_file_suffixes) == 0:
        raise AssertionError(
            "At least one suffix for the to-be-read annotation files should be given!"
        )

    ann_lines = []
    for suffix in annotation_file_suffixes:
        annotation_file = txt_file.with_suffix(suffix)
        try:
            with annotation_file.open() as f:
                ann_lines.extend(f.readlines())
        except Exception:
            continue

    example["text_bound_annotations"] = []
    example["events"] = []
    example["relations"] = []
    example["equivalences"] = []
    example["attributes"] = []
    example["normalizations"] = []

    if parse_notes:
        example["notes"] = []

    for line in ann_lines:
        line = line.strip()
        if not line:
            continue

        if line.startswith("T"):  # Text bound
            ann = {}
            fields = line.split("\t")

            ann["id"] = fields[0]
            ann["type"] = fields[1].split()[0]
            ann["offsets"] = []
            span_str = remove_prefix(fields[1], (ann["type"] + " "))
            text = fields[2]
            for span in span_str.split(";"):
                start, end = span.split()
                ann["offsets"].append([int(start), int(end)])

            # Heuristically split text of discontiguous entities into chunks
            ann["text"] = []
            if len(ann["offsets"]) > 1:
                i = 0
                for start, end in ann["offsets"]:
                    chunk_len = end - start
                    ann["text"].append(text[i : chunk_len + i])
                    i += chunk_len
                    while i < len(text) and text[i] == " ":
                        i += 1
            else:
                ann["text"] = [text]

            example["text_bound_annotations"].append(ann)

        elif line.startswith("E"):
            ann = {}
            fields = line.split("\t")

            ann["id"] = fields[0]

            ann["type"], ann["trigger"] = fields[1].split()[0].split(":")

            ann["arguments"] = []
            for role_ref_id in fields[1].split()[1:]:
                argument = {
                    "role": (role_ref_id.split(":"))[0],
                    "ref_id": (role_ref_id.split(":"))[1],
                }
                ann["arguments"].append(argument)

            example["events"].append(ann)

        elif line.startswith("R"):
            ann = {}
            fields = line.split("\t")

            ann["id"] = fields[0]
            ann["type"] = fields[1].split()[0]

            ann["head"] = {
                "role": fields[1].split()[1].split(":")[0],
                "ref_id": fields[1].split()[1].split(":")[1],
            }
            ann["tail"] = {
                "role": fields[1].split()[2].split(":")[0],
                "ref_id": fields[1].split()[2].split(":")[1],
            }

            example["relations"].append(ann)

        # '*' seems to be the legacy way to mark equivalences,
        # but I couldn't find any info on the current way
        # this might have to be adapted dependent on the brat version
        # of the annotation
        elif line.startswith("*"):
            ann = {}
            fields = line.split("\t")

            ann["id"] = fields[0]
            ann["ref_ids"] = fields[1].split()[1:]

            example["equivalences"].append(ann)

        elif line.startswith("A") or line.startswith("M"):
            ann = {}
            fields = line.split("\t")

            ann["id"] = fields[0]

            info = fields[1].split()
            ann["type"] = info[0]
            ann["ref_id"] = info[1]

            if len(info) > 2:
                ann["value"] = info[2]
            else:
                ann["value"] = ""

            example["attributes"].append(ann)

        elif line.startswith("N"):
            ann = {}
            fields = line.split("\t")

            ann["id"] = fields[0]
            ann["text"] = fields[2]

            info = fields[1].split()

            ann["type"] = info[0]
            ann["ref_id"] = info[1]
            ann["resource_name"] = info[2].split(":")[0]
            ann["cuid"] = info[2].split(":")[1]
            example["normalizations"].append(ann)

        elif parse_notes and line.startswith("#"):
            ann = {}
            fields = line.split("\t")

            ann["id"] = fields[0]
            ann["text"] = fields[2] if len(fields) == 3 else BigBioValues.NULL

            info = fields[1].split()

            ann["type"] = info[0]
            ann["ref_id"] = info[1]
            example["notes"].append(ann)

    return example


def brat_parse_to_bigbio_kb(brat_parse: Dict) -> Dict:
    """
    Transform a brat parse (conforming to the standard brat schema) obtained with
    `parse_brat_file` into a dictionary conforming to the `bigbio-kb` schema (as defined in ../schemas/kb.py)
    :param brat_parse:
    """

    unified_example = {}

    # Prefix all ids with document id to ensure global uniqueness,
    # because brat ids are only unique within their document
    id_prefix = brat_parse["document_id"] + "_"

    # identical
    unified_example["document_id"] = brat_parse["document_id"]
    unified_example["passages"] = [
        {
            "id": id_prefix + "_text",
            "type": "abstract",
            "text": [brat_parse["text"]],
            "offsets": [[0, len(brat_parse["text"])]],
        }
    ]

    # get normalizations
    ref_id_to_normalizations = defaultdict(list)
    for normalization in brat_parse["normalizations"]:
        ref_id_to_normalizations[normalization["ref_id"]].append(
            {
                "db_name": normalization["resource_name"],
                "db_id": normalization["cuid"],
            }
        )

    # separate entities and event triggers
    unified_example["events"] = []
    non_event_ann = brat_parse["text_bound_annotations"].copy()
    for event in brat_parse["events"]:
        event = event.copy()
        event["id"] = id_prefix + event["id"]
        trigger = next(
            tr
            for tr in brat_parse["text_bound_annotations"]
            if tr["id"] == event["trigger"]
        )
        if trigger in non_event_ann:
            non_event_ann.remove(trigger)
        event["trigger"] = {
            "text": trigger["text"].copy(),
            "offsets": trigger["offsets"].copy(),
        }
        for argument in event["arguments"]:
            argument["ref_id"] = id_prefix + argument["ref_id"]

        unified_example["events"].append(event)

    unified_example["entities"] = []
    anno_ids = [ref_id["id"] for ref_id in non_event_ann]
    for ann in non_event_ann:
        entity_ann = ann.copy()
        entity_ann["id"] = id_prefix + entity_ann["id"]
        entity_ann["normalized"] = ref_id_to_normalizations[ann["id"]]
        unified_example["entities"].append(entity_ann)

    # massage relations
    unified_example["relations"] = []
    skipped_relations = set()
    for ann in brat_parse["relations"]:
        if (
            ann["head"]["ref_id"] not in anno_ids
            or ann["tail"]["ref_id"] not in anno_ids
        ):
            skipped_relations.add(ann["id"])
            continue
        unified_example["relations"].append(
            {
                "arg1_id": id_prefix + ann["head"]["ref_id"],
                "arg2_id": id_prefix + ann["tail"]["ref_id"],
                "id": id_prefix + ann["id"],
                "type": ann["type"],
                "normalized": [],
            }
        )
    if len(skipped_relations) > 0:
        example_id = brat_parse["document_id"]
        logger.info(
            f"Example:{example_id}: The `bigbio_kb` schema allows `relations` only between entities."
            f" Skip (for now): "
            f"{list(skipped_relations)}"
        )

    # get coreferences
    unified_example["coreferences"] = []
    for i, ann in enumerate(brat_parse["equivalences"], start=1):
        is_entity_cluster = True
        for ref_id in ann["ref_ids"]:
            if not ref_id.startswith("T"):  # not textbound -> no entity
                is_entity_cluster = False
            elif ref_id not in anno_ids:  # event trigger -> no entity
                is_entity_cluster = False
        if is_entity_cluster:
            entity_ids = [id_prefix + i for i in ann["ref_ids"]]
            unified_example["coreferences"].append(
                {"id": id_prefix + str(i), "entity_ids": entity_ids}
            )
    return unified_example


In [ ]:
# coding=utf-8
# Copyright 2022 The HuggingFace Datasets Authors and the current dataset script contributor.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
Relation Extraction corpus with multiple entity types (e.g., gene/protein,
disease, chemical) and relation pairs (e.g., gene-disease; chemical-chemical),
on a set of 600 PubMed articles
"""

import itertools
import os
from collections import defaultdict
from typing import Dict, List, Tuple

import datasets
from bioc import pubtator

#from load_biored import train
'''from .bigbiohub import kb_features
from .bigbiohub import BigBioConfig
from .bigbiohub import Tasks'''

_LANGUAGES = ["English"]
_PUBMED = True
_LOCAL = False
_CITATION = """\
@article{DBLP:journals/corr/abs-2204-04263,
  author    = {Ling Luo and
               Po{-}Ting Lai and
               Chih{-}Hsuan Wei and
               Cecilia N. Arighi and
               Zhiyong Lu},
  title     = {BioRED: {A} Comprehensive Biomedical Relation Extraction Dataset},
  journal   = {CoRR},
  volume    = {abs/2204.04263},
  year      = {2022},
  url       = {https://doi.org/10.48550/arXiv.2204.04263},
  doi       = {10.48550/arXiv.2204.04263},
  eprinttype = {arXiv},
  eprint    = {2204.04263},
  timestamp = {Wed, 11 May 2022 15:24:37 +0200},
  biburl    = {https://dblp.org/rec/journals/corr/abs-2204-04263.bib},
  bibsource = {dblp computer science bibliography, https://dblp.org}
}
"""

_DATASETNAME = "biored"
_DISPLAYNAME = "BioRED"

_DESCRIPTION = """\
Relation Extraction corpus with multiple entity types (e.g., gene/protein,
disease, chemical) and relation pairs (e.g., gene-disease; chemical-chemical),
on a set of 600 PubMed articles
"""

_HOMEPAGE = "https://ftp.ncbi.nlm.nih.gov/pub/lu/BioRED/"

_LICENSE = "License information unavailable"

_URLS = {
    _DATASETNAME: "https://ftp.ncbi.nlm.nih.gov/pub/lu/BioRED/BIORED.zip",
}

_SUPPORTED_TASKS = [Tasks.NAMED_ENTITY_RECOGNITION, Tasks.NAMED_ENTITY_DISAMBIGUATION, Tasks.RELATION_EXTRACTION]

_SOURCE_VERSION = "1.0.0"

_BIGBIO_VERSION = "1.0.0"

logger = datasets.utils.logging.get_logger(__name__)


class BioredDataset(datasets.GeneratorBasedBuilder):
    """Relation Extraction corpus with multiple entity types (e.g., gene/protein, disease, chemical) and
    relation pairs (e.g., gene-disease; chemical-chemical), on a set of 600 PubMed articles"""

    # For bigbio_kb, this dataset uses a naming convention as
    # uid_[title/abstract/relation/entity_id]_[entity/relation_uid]

    SOURCE_VERSION = datasets.Version(_SOURCE_VERSION)
    BIGBIO_VERSION = datasets.Version(_BIGBIO_VERSION)

    BUILDER_CONFIGS = [
        BigBioConfig(
            name=_DATASETNAME + "_source",
            version=SOURCE_VERSION,
            description=_DATASETNAME + " source schema",
            schema="source",
            subset_id=_DATASETNAME,
        ),
        BigBioConfig(
            name=_DATASETNAME + "_bigbio_kb",
            version=BIGBIO_VERSION,
            description=_DATASETNAME + " BigBio schema",
            schema="bigbio_kb",
            subset_id=_DATASETNAME,
        ),
    ]

    DEFAULT_CONFIG_NAME = _DATASETNAME + "_source"

    TYPE_TO_DATABASE = {
        "CellLine": "Cellosaurus",
        "ChemicalEntity": "MESH",
        "DiseaseOrPhenotypicFeature": "MESH",  # Some diseases are normalized to OMIM (~ handled by special rules)
        "GeneOrGeneProduct": "NCBIGene",
        "OrganismTaxon": "NCBITaxon",
        "SequenceVariant": "dbSNP",  # Not all variants are normalized to dbSNP (~ handled by special rules)
    }

    def _info(self) -> datasets.DatasetInfo:

        if self.config.schema == "source":

            features = datasets.Features(
                {
                    "pmid": datasets.Value("string"),
                    "passages": [
                        {
                            "type": datasets.Value("string"),
                            "text": datasets.Sequence(datasets.Value("string")),
                            "offsets": datasets.Sequence([datasets.Value("int32")]),
                        }
                    ],
                    "entities": [
                        {
                            "text": datasets.Sequence(datasets.Value("string")),
                            "offsets": datasets.Sequence([datasets.Value("int32")]),
                            "concept_id": datasets.Value("string"),
                            "semantic_type_id": datasets.Value("string"),
                        }
                    ],
                    "relations": [
                        {
                            "novel": datasets.Value("string"),
                            "type": datasets.Value("string"),
                            "concept_1": datasets.Value("string"),
                            "concept_2": datasets.Value("string"),
                        }
                    ],
                }
            )

        elif self.config.schema == "bigbio_kb":
            features = kb_features

        return datasets.DatasetInfo(
            description=_DESCRIPTION,
            features=features,
            homepage=_HOMEPAGE,
            license=str(_LICENSE),
            citation=_CITATION,
        )

    def _split_generators(self, dl_manager) -> List[datasets.SplitGenerator]:
        """Returns SplitGenerators."""

        urls = _URLS[_DATASETNAME]
        data_dir = dl_manager.download_and_extract(urls)

        return [
            datasets.SplitGenerator(
                name=datasets.Split.TRAIN,
                # Whatever you put in gen_kwargs will be passed to _generate_examples
                gen_kwargs={
                    "filepath": os.path.join(data_dir, "BioRED", "Train.PubTator"),
                    "split": "train",
                },
            ),
            datasets.SplitGenerator(
                name=datasets.Split.TEST,
                gen_kwargs={
                    "filepath": os.path.join(data_dir, "BioRED", "Test.PubTator"),
                    "split": "test",
                },
            ),
            datasets.SplitGenerator(
                name=datasets.Split.VALIDATION,
                gen_kwargs={
                    "filepath": os.path.join(data_dir, "BioRED", "Dev.PubTator"),
                    "split": "dev",
                },
            ),
        ]

    def _generate_examples(self, filepath, split: str) -> Tuple[int, Dict]:
        """Yields examples as (key, example) tuples."""

        if self.config.schema == "source":
            with open(filepath, "r", encoding="utf8") as fstream:
                for raw_document in self.generate_raw_docs(fstream):
                    document = self.parse_raw_doc(raw_document)
                    yield document["pmid"], document

        elif self.config.schema == "bigbio_kb":
            with open(filepath, "r", encoding="utf8") as fstream:
                uid = itertools.count(0)
                for raw_document in self.generate_raw_docs(fstream):
                    document = self.parse_raw_doc(raw_document)
                    pmid = str(document.pop("pmid"))
                    document["id"] = str(next(uid))
                    document["document_id"] = pmid

                    # Parse entities
                    entities = []
                    entity_id_to_mentions = defaultdict(list)  # Maps database ids to mention ids
                    for i, entity in enumerate(document["entities"]):
                        internal_id = pmid + "_" + str(i)

                        # Some entities are normalized to multiple database ids, therefore we
                        # may have multiple identifiers per mention
                        normalized_entity_ids = []
                        for database_id in entity["concept_id"].split(","):
                            database_id = database_id.strip()
                            entity_type = entity["semantic_type_id"]

                            # First check special db_name and database id assignment rules
                            if entity_type == "DiseaseOrPhenotypicFeature" and database_id.lower().startswith("omim"):
                                db_name = "OMIM"
                                database_id = database_id.split(":")[-1]
                            elif entity_type == "SequenceVariant" and not database_id.startswith("rs"):
                                db_name = "custom"

                            # If no special rule applies -> just take the default db_name for the entity type
                            else:
                                db_name = self.TYPE_TO_DATABASE[entity_type]

                            normalized_entity_ids.append({"db_name": db_name, "db_id": database_id})
                            entity_id_to_mentions[database_id].append(internal_id)

                        entities.append(
                            {
                                "id": internal_id,
                                "type": entity_type,
                                "text": entity["text"],
                                "normalized": normalized_entity_ids,
                                "offsets": entity["offsets"],
                            }
                        )

                    # BioRed provides abstract-level annotations for entity-linked relation pairs rather than
                    # materializing links between all surface form mentions of relation. For example document 11009181
                    # in train has (Positive_Correlation, D007980, D004409). Analogous to BC5CDR we enumerate all
                    # mention pairs concerning the entities in the triple.
                    relations = []
                    rel_uid = itertools.count(0)
                    for relation in document["relations"]:
                        head_mentions = entity_id_to_mentions[str(relation["concept_1"])]
                        tail_mentions = entity_id_to_mentions[str(relation["concept_2"])]

                        for head, tail in itertools.product(head_mentions, tail_mentions):
                            relations.append(
                                {
                                    "id": document["id"] + "_relation_" + str(next(rel_uid)),
                                    "type": relation["type"],
                                    "arg1_id": head,
                                    "arg2_id": tail,
                                    "normalized": [],
                                }
                            )

                    for passage in document["passages"]:
                        passage["id"] = document["id"] + "_" + passage["type"]

                    document["entities"] = entities
                    document["relations"] = relations
                    document["events"] = []
                    document["coreferences"] = []

                    yield document["document_id"], document

    def generate_raw_docs(self, fstream):
        """
        Given a filestream, this function yields documents from it
        """
        raw_document = []
        for line in fstream:
            if line.strip():
                raw_document.append(line.strip())
            elif raw_document:
                yield raw_document
                raw_document = []
        if raw_document:
            yield raw_document

    def parse_raw_doc(self, raw_doc):
        pmid, _, title = raw_doc[0].split("|")
        pmid = int(pmid)
        _, _, abstract = raw_doc[1].split("|")
        passages = [
            {"type": "title", "text": [title], "offsets": [[0, len(title)]]},
            {
                "type": "abstract",
                "text": [abstract],
                "offsets": [[len(title) + 1, len(title) + len(abstract) + 1]],
            },
        ]
        entities = []
        relations = []
        for line in raw_doc[2:]:
            mentions = line.split("\t")
            (_pmid, _type_ind, *rest) = mentions
            if _type_ind in [
                "Positive_Correlation",
                "Association",
                "Negative_Correlation",
                "Bind",
                "Conversion",
                "Cotreatment",
                "Cause",
                "Comparison",
                "Drug_Interaction",
            ]:
                # Relations handled here
                relation_type = _type_ind
                concept_1, concept_2, novel = rest
                relation = {
                    "type": relation_type,
                    "concept_1": concept_1,
                    "concept_2": concept_2,
                    "novel": novel,
                }
                relations.append(relation)
            elif _type_ind.isnumeric():
                # Entities handled here
                start_idx = _type_ind
                end_idx, mention, semantic_type_id, entity_ids = rest
                entities.append(
                    {
                        "offsets": [[int(start_idx), int(end_idx)]],
                        "text": [mention],
                        "semantic_type_id": semantic_type_id,
                        "concept_id": entity_ids,
                    }
                )
            else:
                logger.warn(f"Skipping annotation in Document ID: {_pmid}. Unexpected format")
        return {
            "pmid": pmid,
            "passages": passages,
            "entities": entities,
            "relations": relations,
        }



In [ ]:
import random
import torch
import numpy as np
from transformers import BertTokenizer, BertModel, EarlyStoppingCallback
from sklearn.metrics.pairwise import cosine_similarity
#from BioRED_python_download.biored import BioredDataset #TODO: If running on GoogleColab, comment-out this line and uncomment import dataset from Google Drive below.
from transformers import DistilBertModel, DistilBertTokenizer, DistilBertForSequenceClassification, DataCollatorWithPadding
from nltk.tokenize import word_tokenize
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DistilBertModel, DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification
import itertools
from datasets import Dataset
import evaluate
from collections import Counter
from imblearn.under_sampling import RandomUnderSampler
import wandb
wandb.login(key="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")


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True  # same seed at GPU as well
    torch.backends.cudnn.benchmark = False  # same seed at GPU as well


# On Google Colab if loading dataset from Google Drive
'''from google.colab import drive
drive.mount('/content/drive')
import os
GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'Colab Notebooks'
GOOGLE_DRIVE_PATH = os.path.join('drive', 'My Drive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
print(os.listdir(GOOGLE_DRIVE_PATH))'''

# On Google Colab if loading dataset from Google Drive
#DATA_DIR     = os.path.join(GOOGLE_DRIVE_PATH, 'BioRED_python_download') # for my Google Colab
'''DATA_DIR     = os.path.join(GOOGLE_DRIVE_PATH, 'BioRED_python_download (1)') # for uni Google Colab
CHECKPOINT_DIR = os.path.join(GOOGLE_DRIVE_PATH,'checkpoints')'''


In [ ]:
# Select existing config
builder = BioredDataset(config_name="biored_bigbio_kb")

# Download and prepare the dataset
builder.download_and_prepare()

# Load dataset splits. That is, NO NEED FOR SEED (IN DATA SPLIT, elsewhere SEED is needed) because we are using pre-defined splits from BioRED
dataset = builder.as_dataset()

train_data = dataset["train"]
test_data = dataset["test"]
val_data = dataset["validation"]

#TODO: Remember to change DEBUG to False if running experiments.
DEBUG = False
if DEBUG:
    train_data = train_data.select(range(32))
    val_data = val_data.select(range(16))
    test_data = test_data.select(range(8))
    print("train_set length DEBUGGING:", len(train_data), "val_set length DEBUGGING:", len(val_data), "test_set length DEBUGGING:", len(test_data))

print("Column names:\n", train_data.column_names)
print("Features:\n", train_data.features)
print("Entities:\n", train_data.features["entities"])
print("Number of columns:", train_data.num_columns)

print("First data sample in train:\n", train_data[0])

sample = train_data[0]
print(f"Title/Abstract: {sample['passages'][0]['text']}")
print(f"First Entity: {sample['entities'][0]}")
print(f"First Relation: {sample['relations'][0]}")

'''print("First data sample in train (line by line):")
print(train_data[0]["id"])
print(train_data[0]["document_id"])
print(train_data[0]["passages"])
print(train_data[0]["entities"])
print(train_data[0]["events"])
print(train_data[0]["coreferences"])
print(train_data[0]["relations"])'''

print("Size of training subset (i.e. number of data samples):", train_data.num_rows)
print("Size of validation subset (i.e. number of data samples):", val_data.num_rows)
print("Size of testing subset (i.e. number of data samples):", test_data.num_rows)


In [ ]:
# 1. Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased', add_prefix_space=True)
#tokenizer = BertTokenizer.from_pretrained('google-bert/bert-base-uncased', add_prefix_space=True)

# Map BioRED entity types to ID integers.
# This code was inspired by https://huggingface.co/docs/transformers/tasks/token_classification, the huggingFace tutorial on how to perform NER with BERT
# id2label and label2id inspired by https://github.com/huggingface/notebooks/blob/main/examples/token_classification-tf.ipynb
label_list = [ "O",
               "B-ChemicalEntity", "I-ChemicalEntity",
               "B-DiseaseOrPhenotypicFeature","I-DiseaseOrPhenotypicFeature",
               "B-GeneOrGeneProduct","I-GeneOrGeneProduct",
               "B-SequenceVariant","I-SequenceVariant",
               "B-OrganismTaxon","I-OrganismTaxon",
               "B-CellLine", "I-CellLine"
               ]#["O", "B-Gene", "I-Gene", "B-Disease", "I-Disease", "B-Chemical", "I-Chemical"]  # Simplified for example. #["GeneOrGeneProduct", "DiseaseOrPhenotypicFeature", "ChemicalEntity", "SequenceVariant", "OrganismTaxon", "CellLine"]
label_to_id = {l: i for i, l in enumerate(label_list)}
id_to_label={i: l for i, l in enumerate(label_list)}

# Map BioRED relation types to ID integers
relation_list = [
    "None", #not in original BioRED dataset, but added for logic to determine label
    "Association",
    "Bind",
    "Comparison",
    "Conversion",
    "Cotreatment",
    "Drug_Interaction",
    "Negative_Correlation",
    "Positive_Correlation"
]
relation_to_id = {r: i for i, r in enumerate(relation_list)}
id_to_relation = {i: r for i, r in enumerate(relation_list)}

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer) # only needed for token-level classification tasks, NOT sequence-level classification task. That is, the data_collator is needed for NER, but NOT for RE. The default DataCollatorWithPadding called by Trainer is all RE needs for padding )

def preprocess_biored_for_ner(examples):
    # BioRED KB has 'passages' containing the full text
    texts = [
        " ".join(
            passage_text
            for passage in p
            for passage_text in passage["text"]
        )
        for p in examples["passages"]
    ]

    tokenized_inputs = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_offsets_mapping=True  # CRITICAL for alignment
    )

    labels = []
    for i, document_entities in enumerate(examples["entities"]):
        offset_mapping = tokenized_inputs["offset_mapping"][i]
        doc_labels = np.zeros(len(offset_mapping), dtype=int)

        for entity in document_entities:
            start_char = entity["offsets"][0][0]
            end_char = entity["offsets"][0][1]
            entity_type = entity["type"]
            if entity_type not in [
                "ChemicalEntity",
                "DiseaseOrPhenotypicFeature",
                "GeneOrGeneProduct",
                "SequenceVariant",
                "OrganismTaxon",
                "CellLine"
            ]:
                print(entity_type + " " + str(start_char) + " " + str(end_char))

            # Map tokens to labels based on character offsets
            inside = False
            for idx, (start, end) in enumerate(offset_mapping):
                if start is None or start == end: #if start == end:  # Skip special tokens like [CLS] or [PAD]
                    doc_labels[idx] = -100  # Standard ignore index for loss
                    continue

                if not (end <= start_char or start >= end_char): #if start >= start_char and end <= end_char
                    # Logic for B- and I- tags
                    if not inside:
                        prefix = "B-"
                        inside = True
                    else:
                        prefix = "I-"
                    label_str = f"{prefix}{entity_type}"

                    if label_str in label_to_id:
                        doc_labels[idx] = label_to_id[label_str]
                    else:
                        raise ValueError(f"Unknown label: {label_str}")
        labels.append(doc_labels.tolist())

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


def preprocess_biored_for_re(example):
    #none_keep_rate= 0.1
    entities = example["entities"]
    existing_relations = {(r["arg1_id"], r["arg2_id"]): r["type"] for r in example["relations"]}

    for r in example["relations"]:
        #print(r['type'])
        if r['type'] not in relation_list:
            print("RELATION LIST TEST, relation missing:\n", r, "\n ", r['type'])

    #Alternative to be tried out if for training current implementation of method returning a dictionary in the way it currently does (even though as per HuggingFace it does need to return a dictionary) causes problems
    texts = []
    labels = []

    context = " ".join(example["passages"][0]["text"]) #context = " ".join(example["passages"][0]["text"])  # Simple abstract-level context
    # Generate every possible combination of 2 entities in the document
    for ent1, ent2 in itertools.combinations(entities, 2):
        pair = (ent1["id"], ent2["id"])
        reverse_pair = (ent2["id"], ent1["id"])

        # Determine label (Positive relation or 'None' for Negative Sampling)
        relation_type = "None"
        if pair in existing_relations:
            relation_type = existing_relations[pair]
        elif reverse_pair in existing_relations:
            relation_type = existing_relations[reverse_pair]

        # Create a text representation with Entity Markers and feed entity pair as a sentence pair to the tokenizer
        # "Gene [E1] Factor-6 [/E1] is linked to [E2] Diabetes [/E2]"
        entity_pair_str = f"{ent1['text'][0]} [SEP] {ent2['text'][0]}"

        '''relation_none_counts = Counter(relation_type == "None")
        relation_everything_else_counts = Counter(relation_type != "None")'''
        '''if relation_type == "None" and (relation_none_counts.values()/len(train_re_data['labels'])*100) > (relation_everything_else_counts.values()/len(train_re_data['labels'])*100):'''
        '''if relation_type == "None" and (relation_none_counts > relation_everything_else_counts):
            continue'''

        texts.append((context, entity_pair_str))  # pair-level tokenization, unlike for NER, which requires document-level tokenization and hence, why input is tokenized at a different stage / before preparing the data for the NER module
        if relation_type not in relation_to_id:
            raise ValueError(f"Unknown relation type: {relation_type}")
        labels.append(relation_to_id[relation_type])  # e.g. 1

        # Tokenize all pairs for this document at once
    tokenized = tokenizer(
        [t[0] for t in texts],   # contexts
        [t[1] for t in texts],   # entity pairs
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokenized["labels"] = labels
    return tokenized








def flatten_re_dataset(re_mapped):
    """
    preprocess_biored_for_re returns variable-length lists per document
    (one entry per entity pair) such that .map function can be applied to preprocess_biored_for_re, but this is incompatible with Trainer, which expexts a different format. This flattens them into individual samples.
    """
    all_input_ids      = []
    all_attention_mask = []
    all_token_type_ids = []
    all_labels         = []

    for sample in re_mapped:
        ids = sample["input_ids"]
        # Each document produces a list of lists (one per entity pair)
        if isinstance(ids[0], list):
            all_input_ids.extend(ids)
            all_attention_mask.extend(sample["attention_mask"])
            if "token_type_ids" in sample:
                all_token_type_ids.extend(sample["token_type_ids"])
            all_labels.extend(sample["labels"])
        else:
            all_input_ids.append(ids)
            all_attention_mask.append(sample["attention_mask"])
            if "token_type_ids" in sample:
                all_token_type_ids.append(sample["token_type_ids"])
            all_labels.append(sample["labels"])

    flat = {
        "input_ids":      all_input_ids,
        "attention_mask": all_attention_mask,
        "labels":         all_labels
    }
    if all_token_type_ids:
        flat["token_type_ids"] = all_token_type_ids

    return Dataset.from_dict(flat)


In [ ]:
if __name__ == "__main__":
    train_ner_data = train_data.map(preprocess_biored_for_ner, batched=True) # apply to train subset
    val_ner_data = val_data.map(preprocess_biored_for_ner, batched=True)  # apply to val subset
    test_ner_data = test_data.map(preprocess_biored_for_ner, batched=True)  # apply to test subset

    #print(train_ner_data[0])
    #print("NER data preprocessing complete")

    train_re_data = train_data.map(preprocess_biored_for_re) #, batched=True) # apply to train subset
    val_re_data = val_data.map(preprocess_biored_for_re) #, batched=True) # apply to val subset
    test_re_data = test_data.map(preprocess_biored_for_re) #, batched=True)  # apply to test subset

    #print(train_re_data[0])
    #print("RE data preprocessing complete")

    train_re_data = flatten_re_dataset(train_re_data)
    val_re_data   = flatten_re_dataset(val_re_data)
    test_re_data  = flatten_re_dataset(test_re_data)

    X_train = np.array(train_re_data["input_ids"])
    y_train = np.array(train_re_data["labels"])

    rus = RandomUnderSampler(
    sampling_strategy={relation_to_id["None"]: 20000},
    random_state=42
    )
    X_resampled, y_resampled = rus.fit_resample(X_train, y_train)

    # Get the indices RUS selected so we can apply them to other columns
    indices = rus.sample_indices_

    # Rebuild the dataset using those indices — keeps all columns in sync
    train_re_data = Dataset.from_dict({
    "input_ids":      X_resampled.tolist(),
    "attention_mask": [train_re_data["attention_mask"][i] for i in indices],
    "token_type_ids": [train_re_data["token_type_ids"][i] for i in indices],
    "labels":         y_resampled.tolist()
    })

    print(Counter(y_resampled))

    label_counts = Counter(train_re_data["labels"])
    print("Label counts AFTER RESAMPLING:", label_counts.total())
    print("Label distribution AFTER RESAMPLING:")
    for label_id, count in sorted(label_counts.items()):
        rel_name = id_to_relation[label_id]
        print(f"  {rel_name} (id={label_id}): {count}. That is, {count/len(train_re_data['labels'])*100:.2f}%")

    '''label_counts = Counter(train_re_data["labels"])
    print("Label counts (total number of relationships associated with a label or with none):", label_counts.total())
    print("Label distribution (with None / negative samples):")
    for label_id, count in sorted(label_counts.items()):
        rel_name = id_to_relation[label_id]
        print(f"  {rel_name} (id={label_id}): {count}. That is, {count/len(train_re_data['labels'])*100:.2f}% of the training set.")


    X_train = np.array(train_re_data["input_ids"])   # shape: (num_samples, 512)
    y_train = np.array(train_re_data["labels"])

    # Only undersample None, leave everything else untouched
    rus = RandomUnderSampler(
        sampling_strategy={relation_to_id["None"]: 10000},  # Only undersample None, leave everything else untouched and keep 10k None samples
        random_state=42
    )
    X_resampled, y_resampled = rus.fit_resample(X_train, y_train)
    train_re_data["input_ids"]=X_resampled
    train_re_data["labels"]=y_resampled
    print(Counter(y_resampled))  # all classes now equal to smallest class

    label_counts = Counter(train_re_data["labels"])
    print("Label counts AFTER RESAMPLING:", label_counts.total())
    print("Label distribution (with None / negative samples) AFTER RESAMPLING:")
    for label_id, count in sorted(label_counts.items()):
        rel_name = id_to_relation[label_id]
        print(f"  {rel_name} (id={label_id}): {count}. That is, {count/len(train_re_data['labels'])*100:.2f}% of the training set.")'''

    train_ner_data.set_format(type="torch")#, output_all_columns=True) # convert it to tensor for torch and Transformers to be able to handle the training subset for NER
    val_ner_data.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the val subset for NER
    test_ner_data.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the test subset for NER

    train_re_data.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the training subset for RE
    val_re_data.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the val subset for RE
    test_re_data.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the test subset for RE

    print(train_ner_data.column_names)
    print(train_re_data.column_names)

    # Remove offset_mapping after preprocessing — model doesn't need it
    train_ner_data = train_ner_data.remove_columns(["offset_mapping"])
    val_ner_data = val_ner_data.remove_columns(["offset_mapping"])
    test_ner_data = test_ner_data.remove_columns(["offset_mapping"])

    print(train_ner_data.column_names)
    print(train_re_data.column_names)

In [ ]:
# use BERT for NER
ner_model = AutoModelForTokenClassification.from_pretrained( #no explicit word embeddings are created because these are automatically created and used internally by BERT
    "google-bert/bert-base-uncased",
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
).to(DEVICE)

#https://huggingface.co/docs/transformers/v5.8.0/en/main_classes/trainer#transformers.TrainingArguments
def train_ner(train_ner_data_arg, val_ner_data_arg):
    ner_training_args = TrainingArguments(
        output_dir="./ner_results",
        per_device_train_batch_size= 16, #train_config_ner['batch_size'],
        per_device_eval_batch_size= 16, #train_config_ner['eval_batch_size'],
        num_train_epochs=15, #train_config_ner['num_epochs'],
        #max_steps=25, # as max_steps rule of thumb is number of samples (400 in train) divided by per_device_train_batch_size(16)xnum_gpus#defult value for now,
        eval_strategy="epoch",
        save_strategy="best", #train_config_ner['save_strategy'],
        logging_strategy="epoch",
        learning_rate=2e-5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",#train_config_ner['metric_for_best_model'],
        seed=42, # DataCollatorForLanguageModeling includes seed, but DataCollatorForTokenClassification does NOT. So, we need to set it here.
        report_to="wandb" #,
        #run_name="NER_BERT-based_pipeline_base_uncased"
    )

    ner_trainer = Trainer(
        model=ner_model,
        args=ner_training_args,
        train_dataset=train_ner_data_arg,
        eval_dataset=val_ner_data_arg,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_ner,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.01)]
    )
    ner_trainer.train()
    return ner_trainer


seqeval = evaluate.load("seqeval")
def compute_metrics_ner(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels, zero_division=0)

        # Print per-entity-type breakdown
    for key, value in results.items():
        if isinstance(value, dict):  # per-class results are dicts
            print(f"{key}: {value}")

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [ ]:
# use BERT for RE
re_model = AutoModelForSequenceClassification.from_pretrained(#no explicit word embeddings are created because these are automatically created and used internally by BERT
    "google-bert/bert-base-uncased",
    num_labels=len(relation_list),
    id2label=id_to_relation,
    label2id=relation_to_id
).to(DEVICE)

#print(re_model)

'''re_model = DistilBertForSequenceClassification.from_pretrained(#no explicit word embeddings are created because these are automatically created and used internally by BERT
    "google-bert/bert-base-uncased",
    num_labels=len(relation_list),
    id2label=id_to_relation,
    label2id=relation_to_id
)'''

#https://huggingface.co/docs/transformers/v5.8.0/en/main_classes/trainer#transformers.TrainingArguments: "load_best_model_at_end (bool, optional, defaults to False) — Load the best checkpoint at the end of training. Requires eval_strategy to be set. When enabled, the best checkpoint is always saved (see save_total_limit). When `True`, `save_strategy` must match `eval_strategy` (unless `save_strategy` is `"best"`), and if using `"steps"`, `save_steps` must be a multiple of `eval_steps`."

def train_re(train_re_data_arg, val_re_data_arg):
    #print(train_re_data[0])
    re_training_args = TrainingArguments(
        output_dir="./re_results",
        per_device_train_batch_size= 16, #train_config_re['batch_size'],
        per_device_eval_batch_size= 16, #train_config_re['eval_batch_size'],
        num_train_epochs=15,
        #max_steps=100, #100 for degugging #25, # as max_steps rule of thumb is number of samples (400 in train) divided by per_device_train_batch_size(16)xnum_gpus #defult value for now, 25 # TODO: Remove max_steps entirely WHEN RUNNING EXPERIMENTS
        eval_strategy="epoch",
        save_strategy="best", #train_config_re['save_strategy'],
        logging_strategy="epoch",
        learning_rate=2e-5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",#train_config_re['metric_for_best_model'],
        seed=42, # DataCollatorForLanguageModeling includes seed, but DataCollatorForTokenClassification does NOT. So, we need to set it here.
        report_to="wandb" #,
        #run_name="RE_BERT-based_pipeline_base_uncased"
    )

    re_trainer = Trainer(
        model=re_model,
        args=re_training_args,
        train_dataset=train_re_data_arg,
        eval_dataset=val_re_data_arg,
        processing_class=tokenizer,
        compute_metrics=compute_metrics_re,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.05)]
    )
    re_trainer.train()
    return re_trainer

'''def compute_metrics_re(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    results = seqeval.compute(predictions=predictions, references=labels)
    return results["overall_accuracy"] # because "The compute function needs to receive a tuple (with logits and labels) and has to return a dictionary with string keys (the name of the metric) and float values" https://huggingface.co/docs/transformers/v4.14.1/training'''


#metric = evaluate.load("accuracy")
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
def compute_metrics_re(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
    "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
    "f1": f1.compute(predictions=predictions, references=labels, average="macro")["f1"],
    "precision": precision.compute(predictions=predictions, references=labels, average="macro", zero_division=0)["precision"],
    "recall": recall.compute(predictions=predictions, references=labels, average="macro", zero_division=0)["recall"],
    }
    #return metric.compute(predictions=predictions, references=labels) # because "The compute function needs to receive a tuple (with logits and labels) and has to return a dictionary with string keys (the name of the metric) and float values" https://huggingface.co/docs/transformers/v4.14.1/training. However, code was inspired by https://huggingface.co/docs/evaluate/transformers_integrations since the latter is outdated (deprecated)
    #seqeval cannot be used for RE because Seqeval is a Python framework designed specifically for evaluating sequence labeling tasks, such as Named Entity Recognition (NER), Part-of-Speech (PoS) tagging, and semantic role labeling. It is NOT compatible with AutoModelForSequenceClassification (i.e. RE module in this pipeline) as AutoModelForSequenceClassification is used for document-level classification, seqeval is used for evaluating performance of Transformers in token-level tasks, i.e. NER


In [ ]:
if __name__ == "__main__":

    wandb.init(project="BERT-based_pipeline", name="NER_BERT-based_pipeline_EXPERIMENT_2")
    ner_trainer = train_ner(train_ner_data, val_ner_data)
    wandb.finish()

    wandb.init(project="BERT-based_pipeline", name="RE_BERT-based_pipeline_EXPERIMENT_2")
    re_trainer  = train_re(train_re_data,  val_re_data)
    wandb.finish()

    ner_trainer.save_model("./ner_model_final")
    re_trainer.save_model("./re_model_final")

    print("BERT-based Pipeline; training complete.")

    # like this?

In [ ]:
#Inference of NER modules and of RE module separatedly, and of whole pipeline (inference of RE when input to RE is predictions of NER on the test dataset)

'''from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# 1. Define the directory where your model is saved
ner_model_dir = "./ner_model_final"

# 2. Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(ner_model_dir)
model = AutoModelForTokenClassification.from_pretrained(ner_model_dir)

# 3. Set to evaluation mode for inference
def ner_inference(test_ner_data_arg):
    model.eval()
    inputs = tokenizer(test_ner_data_arg, return_tensors="pt")

    with torch.no_grad(): #disble gradient calculation
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2) # model outputs are logits, so we need to convert them to predictions using argmax
    print(predictions)
'''